# Keypoint Counting Classifiers (KCC)

This notebook demonstrates the setup for **Keypoint Counting Classifiers (KCC)**, a method for few-shot image classification that counts how many learned visual keypoints appear in a query image.

KCC uses [DINOv2](https://github.com/facebookresearch/dinov2) as a backbone for extracting rich visual features from images.

**Reference:**
> Wickstrøm, K., et al. *Keypoint Counting Classifiers*. arXiv:2512.17891, 2024. https://arxiv.org/abs/2512.17891

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision transformers Pillow requests matplotlib --quiet

## 2. Load Example Bird Images from Wikimedia Commons

We load a small set of bird images freely available from [Wikimedia Commons](https://commons.wikimedia.org/).

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

# Bird images from Wikimedia Commons (CC-licensed)
BIRD_IMAGES = {
    "Atlantic Puffin": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/"
        "Atlantic_Puffin_mar08.jpg/320px-Atlantic_Puffin_mar08.jpg"
    ),
    "European Robin": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f3/"
        "Erithacus_rubecula_with_cocked_head.jpg/320px-Erithacus_rubecula_with_cocked_head.jpg"
    ),
    "Blue Tit": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/1/11/"
        "Blue_tit_%28Cyanistes_caeruleus%29.jpg/320px-Blue_tit_%28Cyanistes_caeruleus%29.jpg"
    ),
    "Common Kingfisher": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/9/9e/"
        "Alcedo_atthis_-_Riserva_Naturale_del_Lago_di_Candia.jpg/320px-Alcedo_atthis_-_Riserva_Naturale_del_Lago_di_Candia.jpg"
    ),
}


def load_image_from_url(url: str) -> Image.Image:
    """Download an image from a URL and return a PIL Image."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")


images = {name: load_image_from_url(url) for name, url in BIRD_IMAGES.items()}
print(f"Loaded {len(images)} bird images.")

In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.suptitle("Example Bird Images (Wikimedia Commons)", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Set Up DINOv2 Feature Extraction

[DINOv2](https://github.com/facebookresearch/dinov2) (Oquab et al., 2023) provides powerful self-supervised visual features that serve as the backbone for KCC.
We use the `facebook/dinov2-base` checkpoint from Hugging Face.

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel

DINOV2_MODEL = "facebook/dinov2-base"

processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL)
model = AutoModel.from_pretrained(DINOV2_MODEL)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"DINOv2 model loaded on {device}.")
print(f"Model hidden size: {model.config.hidden_size}")

## 4. Extract DINOv2 Features

We extract patch-level features from each bird image. These dense feature maps form the basis for keypoint detection in KCC.

In [ ]:
import numpy as np


def extract_features(image: Image.Image) -> torch.Tensor:
    """Extract patch-level DINOv2 features from a PIL image.

    Returns a tensor of shape (num_patches, hidden_size).
    """
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # last_hidden_state: (1, num_patches + 1, hidden_size)
    # Index 0 is the [CLS] token; remaining tokens are patch features
    patch_features = outputs.last_hidden_state[0, 1:, :]
    return patch_features.cpu()


features = {name: extract_features(img) for name, img in images.items()}

for name, feat in features.items():
    print(f"{name}: feature shape = {feat.shape}")

## 5. Visualise Patch Features with PCA

We use PCA to project the high-dimensional patch features to 3 dimensions and visualise them as an RGB overlay on each image.

In [ ]:
from sklearn.decomposition import PCA

# Install scikit-learn if needed
try:
    import sklearn  # noqa: F401
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "--quiet"])
    from sklearn.decomposition import PCA  # noqa: F811

patch_size = 14  # DINOv2-base uses 14×14 patches

fig, axes = plt.subplots(2, len(images), figsize=(16, 6))
for col, (name, feat) in enumerate(features.items()):
    feat_np = feat.numpy()  # (num_patches, hidden_size)

    # Fit PCA to 3 components for RGB visualisation
    pca = PCA(n_components=3)
    pca_feat = pca.fit_transform(feat_np)  # (num_patches, 3)

    # Normalise each channel to [0, 1]
    for c in range(3):
        pca_feat[:, c] = (pca_feat[:, c] - pca_feat[:, c].min()) / (
            pca_feat[:, c].max() - pca_feat[:, c].min() + 1e-8
        )

    n_patches = feat_np.shape[0]
    grid_size = int(np.sqrt(n_patches))
    pca_map = pca_feat.reshape(grid_size, grid_size, 3)

    axes[0, col].imshow(images[name])
    axes[0, col].set_title(name, fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(pca_map)
    axes[1, col].set_title("PCA features", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle("DINOv2 Patch Features (PCA → RGB)", fontsize=13)
plt.tight_layout()
plt.show()

## Next Steps

With per-patch DINOv2 features in hand, the KCC pipeline continues by:

1. **Learning keypoints** – clustering or optimising patch-level features to identify distinctive visual parts.
2. **Counting keypoint activations** – for a query image, counting how many of the learned keypoints are present.
3. **Classification** – using keypoint counts as a lightweight, interpretable feature vector for a few-shot classifier.

See the paper for full details: https://arxiv.org/abs/2512.17891